# Đánh giá câu trả lời của LLM — cấu hình chốt (vector + rerank + lọc phiên bản)

Notebook truy hồi (`so_sanh_truy_hoi_dev.ipynb`) chỉ cho biết hệ thống **lấy đúng đoạn** hay không. Notebook này đo **câu trả lời
cuối cùng** (nhận xét tuần 6, ý 4: lưu câu trả lời, evidence, rubric, số token/latency và xử lý lỗi như một outcome).

Chạy trên **máy local** (kernel `venv/bin/python`); **LLM (Ollama), embedding và reranker nằm trên Colab** — chạy
`scripts/setup_server_v2.sh` rồi chép `OLLAMA_BASE_URL`, `MODEL_SERVER_URL` vào `.env`. Máy local không dùng GPU.

Ba lớp đánh giá:
1. **Tự động** (tất định): ngữ cảnh đưa vào LLM có đoạn đúng không; con số trong đáp án có xuất hiện trong câu trả lời không
   (`eval/cham_tu_dong.py`); câu trả lời có phải từ chối không. Chỉ là chỉ số phụ.
2. **Tách lỗi**: ngữ cảnh có đoạn đúng × câu trả lời đúng → biết lỗi nằm ở truy hồi hay ở LLM.
3. **Người chấm** (chuẩn): xuất `cham-tay.csv`, mỗi người chấm một bản theo `docs/rubric-cham-tra-loi.md`; ô cuối tính tỉ lệ
   đúng, bảng tách lỗi và Cohen κ (giữa hai người, giữa chỉ số tự động và người).

- Chỉ chạy **cấu hình chốt**, nên không kết luận "cấu hình cũ so với mới" cho câu trả lời.
- Bộ câu hỏi: các bộ dev **có đáp án chuẩn** (đợt 2, 3, 4 — 99 câu). **Không chạy tập kiểm chứng** ở đây.
- **Lưu sau từng câu**: tunnel đứt thì sửa `.env`, Restart Kernel, đặt `TEN_LAN_CHAY` = tên thư mục lần trước rồi chạy lại —
  câu đã có câu trả lời được giữ, câu `loi` được chạy lại. Khoảng 99 × 9–15 giây ≈ 15–25 phút.


In [1]:
# ---- Tham số ----
BO_TRA_LOI = ["dev_questions_dot2.json", "dev_questions_dot3.json", "dev_questions_dot4.json"]  # các bộ có `dap_an`
# Câu ngoài phạm vi (dev, không có dieu_dung/dap_an — đo từ chối đúng / bịa): BO_TRA_LOI = ["dev_questions_ngoai.json"]
TEN_LAN_CHAY = None  # None = thư mục mới; đặt tên thư mục cũ trong eval/results/ để chạy tiếp lần bị ngắt / tổng hợp lại

import dataclasses, json, os, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "src" / "config.py").exists(), f"Không thấy src/config.py dưới {ROOT} — mở notebook từ thư mục repo"
os.chdir(ROOT)
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # máy local không dùng GPU: model chạy trên Colab

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)
os.environ["RAG_PROFILE"] = "server"
for k in ("RAG_RERANK", "RAG_LOC_PHIEN_BAN", "RAG_SEED", "RAG_TEMPERATURE", "RAG_THINKING"):
    os.environ.pop(k, None)  # không để biến môi trường ép cấu hình
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "eval")]

from dong_bang import chua_commit as _chua_commit, git
COMMIT, BRANCH, chua_commit = git("rev-parse", "--short", "HEAD"), git("branch", "--show-current"), _chua_commit()
TEN = TEN_LAN_CHAY or (f"danh-gia-tra-loi-{time.strftime('%Y-%m-%d-%H%M')}-{COMMIT}"
                       f"{'-chua-commit' if chua_commit else ''}")
OUT = ROOT / "eval" / "results" / TEN
OUT.mkdir(parents=True, exist_ok=True)
print(f"nhánh {BRANCH} | commit {COMMIT} | kết quả: {OUT.relative_to(ROOT)}")
if chua_commit:
    print("CẢNH BÁO: code/dữ liệu đo có thay đổi chưa commit — số liệu không ứng với commit trên:\n" + chua_commit)


nhánh main | commit a6e63eb | kết quả: eval/results/danh-gia-tra-loi-2026-10-23-2336-a6e63eb-chua-commit
CẢNH BÁO: code/dữ liệu đo có thay đổi chưa commit — số liệu không ứng với commit trên:
M eval/evaluate.py
 M eval/luat_cham.py
 M eval/metrics.py
 M eval/tra_loi_dev.py
?? eval/bao_phu.py
?? eval/cham_tu_dong.py
?? eval/dev_questions_dot4.json
?? eval/dong_bang.py
?? eval/kiem_tra_trung.py


## 1. Kiểm tra Ollama và model server trên Colab


In [2]:
import httpx
from remote_models import goi_server

MODEL_SERVER_URL = os.environ.get("MODEL_SERVER_URL", "").strip()
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "").strip()
if not MODEL_SERVER_URL or not OLLAMA_BASE_URL:
    raise SystemExit("Cần cả OLLAMA_BASE_URL và MODEL_SERVER_URL trong .env (chạy setup_server_v2.sh trên Colab)")
SERVER_HEALTH = goi_server("GET", f"{MODEL_SERVER_URL}/health", timeout=30).json()
for lan in range(3):
    try:
        MO_HINH_OLLAMA = [m["name"] for m in httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=20).raise_for_status().json()["models"]]
        break
    except httpx.HTTPError as e:
        print(f"Ollama lỗi lần {lan + 1}: {type(e).__name__}")
        time.sleep(5)
else:
    raise SystemExit(f"Không gọi được Ollama {OLLAMA_BASE_URL} — kiểm tra tunnel trên Colab")
print("model server:", SERVER_HEALTH, "\nOllama:", MO_HINH_OLLAMA)


model server: {'embed_model': 'AITeamVN/Vietnamese_Embedding', 'reranker_model': 'AITeamVN/Vietnamese_Reranker', 'reranker_max_length': 2048, 'device': 'cuda', 'fp16': True} 
Ollama: ['qwen3:8b']


## 2. Dựng query engine của cấu hình chốt


In [3]:
import logging
logging.basicConfig(level=logging.WARNING)
from llama_index.core import Settings
from llama_index.core.callbacks import CallbackManager
from config import get_config
from index_builder import build_or_load_index
from query_engine import build_query_engine
from tra_loi_dev import DemLLM, tra_loi_mot_cau

cfg0 = get_config()
# src/config.py đọc URL một lần lúc import: kernel đã import trước khi sửa .env thì vẫn gọi tunnel cũ
if cfg0.model_server_url != MODEL_SERVER_URL or cfg0.ollama_base_url.rstrip("/") != OLLAMA_BASE_URL.rstrip("/"):
    raise SystemExit("Kernel đang giữ URL cũ khác .env: Kernel > Restart Kernel rồi chạy lại từ đầu")
cfg = dataclasses.replace(cfg0, streaming=False)
assert cfg.use_rerank and cfg.loc_phien_ban and not cfg.use_hybrid, "phải là cấu hình chốt: vector + rerank + lọc phiên bản"
assert cfg.llm_model_name in MO_HINH_OLLAMA, f"Ollama chưa có {cfg.llm_model_name}"
index = build_or_load_index(cfg)
dem = DemLLM()
Settings.callback_manager = CallbackManager([dem])  # đếm token thật do Ollama trả về
engine = build_query_engine(index, cfg)
k0 = tra_loi_mot_cau(engine, dem, "Một năm học có bao nhiêu học kỳ?")  # khởi động (nạp LLM lên GPU), không tính
print(f"{cfg.llm_model_name}, temperature={cfg.temperature}, seed={cfg.seed}, top_k={cfg.similarity_top_k}, "
      f"rerank top_n={cfg.reranker_top_n} ({cfg.reranker_max_length} token), lọc phiên bản | khởi động: {k0['ket_qua']}")


/mnt/data/1_Projects/sgu-rag-llamaindex/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qwen3:8b, temperature=0.0, seed=42, top_k=10, rerank top_n=3 (2048 token), lọc phiên bản | khởi động: chua_cham


## 3. Sinh câu trả lời (lưu sau từng câu, chạy lại được)


In [4]:
from cham_tu_dong import khop_su_kien, la_tu_choi
from luat_cham import BO_CAU_HOI, chuan_hoa_nhan

LUAT = dict(BO_CAU_HOI)
JSONL = OUT / "tra-loi.jsonl"
da_co = {}
if JSONL.exists():
    for dong in JSONL.read_text(encoding="utf-8").splitlines():
        d = json.loads(dong)
        da_co[(d["bo"], d["id"])] = d  # dòng sau (lần chạy lại) ghi đè dòng trước
ket = []
for bo in BO_TRA_LOI:
    cau_hoi = json.loads((ROOT / "eval" / bo).read_text(encoding="utf-8"))
    print(f"=== {bo} ({len(cau_hoi)} câu) ===", flush=True)
    for i, q in enumerate(cau_hoi, start=1):
        ma = str(q.get("id", i))
        cu = da_co.get((bo, ma))
        if cu and cu["ket_qua"] != "loi":
            ket.append(cu)
            continue
        nhan = chuan_hoa_nhan(q.get("dieu_dung") or "")  # câu ngoài phạm vi không có nhãn Điều
        k = tra_loi_mot_cau(engine, dem, q["question"], LUAT.get(bo), nhan or None)
        d = {"bo": bo, "id": ma, "question": q["question"], "dieu_dung": nhan, "dap_an": q.get("dap_an", ""),
             "loai": q.get("loai"), "pham_vi": q.get("pham_vi", "trong"), **k}
        d["tu_choi"] = la_tu_choi(d["tra_loi"]) if d["tra_loi"] else None
        d["khop_su_kien"] = khop_su_kien(d["dap_an"], d["tra_loi"]) if d["tra_loi"] else "loi"
        with JSONL.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(d, ensure_ascii=False) + "\n")
        ket.append(d)
        print(f"{i}/{len(cau_hoi)} {d['ket_qua']:9} ngữ cảnh đúng={d.get('ngu_canh_co_doan_dung')} "
              f"khớp số={d['khop_su_kien']} {d.get('thoi_gian_s')}s", flush=True)
so_loi = sum(1 for d in ket if d["ket_qua"] == "loi")
print(f"Xong {len(ket)} câu, {so_loi} câu lỗi" + (" — chạy lại ô này để thử lại các câu lỗi" if so_loi else ""))


=== dev_questions_dot2.json (15 câu) ===
1/15 chua_cham ngữ cảnh đúng=True khớp số=du 8.01s
2/15 chua_cham ngữ cảnh đúng=True khớp số=du 11.47s
3/15 chua_cham ngữ cảnh đúng=True khớp số=du 7.88s
4/15 chua_cham ngữ cảnh đúng=True khớp số=du 7.19s
5/15 chua_cham ngữ cảnh đúng=True khớp số=du 9.11s
6/15 chua_cham ngữ cảnh đúng=True khớp số=khong 11.3s
7/15 chua_cham ngữ cảnh đúng=True khớp số=du 7.62s
8/15 chua_cham ngữ cảnh đúng=True khớp số=du 9.29s
9/15 chua_cham ngữ cảnh đúng=True khớp số=du 10.32s
10/15 chua_cham ngữ cảnh đúng=True khớp số=du 10.27s
11/15 chua_cham ngữ cảnh đúng=True khớp số=du 9.61s
12/15 chua_cham ngữ cảnh đúng=True khớp số=thieu 9.15s
13/15 chua_cham ngữ cảnh đúng=True khớp số=du 17.93s
14/15 chua_cham ngữ cảnh đúng=True khớp số=khong_ap_dung 15.78s
15/15 chua_cham ngữ cảnh đúng=True khớp số=khong 8.59s
=== dev_questions_dot3.json (15 câu) ===
1/15 chua_cham ngữ cảnh đúng=True khớp số=du 7.67s
2/15 chua_cham ngữ cảnh đúng=True khớp số=du 8.3s
3/15 chua_cham ngữ cả

## 4. Chỉ số tự động (tạm tính, chưa có người chấm)


In [5]:
import statistics
from collections import Counter
from IPython.display import Markdown, display

TEN_BO = {"dev_questions_dot2.json": "Dev đợt 2 (tiếng Anh)", "dev_questions_dot3.json": "Dev đợt 3",
          "dev_questions_dot4.json": "Dev đợt 4", "dev_questions_ngoai.json": "Dev ngoài phạm vi"}

def ty_le(a, b):
    return f"{a}/{b} ({a / b:.0%})" if b else "—"

def bang_tu_dong(ds):
    dong = ["| Bộ | Số câu | Lỗi | Từ chối | Ngữ cảnh có đoạn đúng | Khớp số: đủ / thiếu / không (trên số câu có số) "
            "| Token vào / ra TB | Giây/câu (trung vị / p90) |", "|---|---|---|---|---|---|---|---|"]
    for bo in BO_TRA_LOI:
        x = [d for d in ds if d["bo"] == bo]
        ok = [d for d in x if d["ket_qua"] != "loi"]
        ks = Counter(d["khop_su_kien"] for d in ok)
        co_so = ks["du"] + ks["thieu"] + ks["khong"]
        tg = sorted(d["thoi_gian_s"] for d in ok if d.get("thoi_gian_s"))
        tv = [d["token_vao"] for d in ok if d.get("token_vao")]
        tr = [d["token_ra"] for d in ok if d.get("token_ra")]
        nc = [d for d in ok if "ngu_canh_co_doan_dung" in d]  # câu ngoài phạm vi không có nhãn để kiểm ngữ cảnh
        dong.append(
            f"| {TEN_BO.get(bo, bo)} | {len(x)} | {len(x) - len(ok)} | {sum(1 for d in ok if d['tu_choi'])} | "
            f"{ty_le(sum(1 for d in nc if d['ngu_canh_co_doan_dung']), len(nc))} | "
            f"{ks['du']} / {ks['thieu']} / {ks['khong']} (trên {co_so}) | "
            f"{round(statistics.mean(tv)) if tv else '—'} / {round(statistics.mean(tr)) if tr else '—'} | "
            f"{statistics.median(tg) if tg else '—'} / {tg[max(0, round(0.9 * len(tg)) - 1)] if tg else '—'} |")
    return "\n".join(dong)

def bang_tach_loi(ds, dung_cua, ten_cot):
    # Ngữ cảnh có đoạn đúng × câu trả lời đúng: lỗi nằm ở truy hồi hay ở LLM
    o = Counter((bool(d.get("ngu_canh_co_doan_dung")), dung_cua(d)) for d in ds)
    return "\n".join([f"| | {ten_cot} | Không đúng |", "|---|---|---|",
                      f"| **Ngữ cảnh có đoạn đúng** | {o[(True, True)]} — hệ thống đúng | {o[(True, False)]} — **lỗi LLM** |",
                      f"| **Ngữ cảnh không có đoạn đúng** | {o[(False, True)]} — đúng dù thiếu căn cứ | "
                      f"{o[(False, False)]} — **lỗi truy hồi** |"])

ngoai = [d for d in ket if d.get("pham_vi") == "ngoai" and d["ket_qua"] != "loi"]
MD_NGOAI = [] if not ngoai else [
    "### Câu ngoài phạm vi",
    f"Từ chối theo mẫu (la_tu_choi): {ty_le(sum(1 for d in ngoai if d['tu_choi']), len(ngoai))}. Câu còn lại phải đọc tay: "
    "trả lời có căn cứ thật trong corpus (nhãn phạm vi cần sửa) hay bịa."]
co_so = [d for d in ket if d["ket_qua"] != "loi" and d["khop_su_kien"] in ("du", "thieu", "khong")]
MD_TU_DONG = "\n\n".join([
    "### Chỉ số tự động theo bộ (không gộp)", bang_tu_dong(ket),
    f"### Tách lỗi tạm tính bằng khớp số ({len(co_so)} câu có con số trong đáp án)",
    bang_tach_loi(co_so, lambda d: d["khop_su_kien"] == "du", "Khớp đủ số"),
    "> Khớp số chỉ là chỉ số phụ: đúng số chưa chắc đúng ý (vd đúng số nhưng sai khóa), và câu định tính không chấm được. "
    "Kết luận dùng bảng của người chấm ở mục 6.", *MD_NGOAI])
display(Markdown(MD_TU_DONG))


### Chỉ số tự động theo bộ (không gộp)

| Bộ | Số câu | Lỗi | Từ chối | Ngữ cảnh có đoạn đúng | Khớp số: đủ / thiếu / không (trên số câu có số) | Token vào / ra TB | Giây/câu (trung vị / p90) |
|---|---|---|---|---|---|---|---|
| Dev đợt 2 (tiếng Anh) | 15 | 0 | 0 | 15/15 (100%) | 11 / 1 / 2 (trên 14) | 3071 / 151 | 9.29 / 15.78 |
| Dev đợt 3 | 15 | 0 | 1 | 15/15 (100%) | 9 / 4 / 0 (trên 13) | 2469 / 157 | 10.11 / 15.99 |
| Dev đợt 4 | 69 | 0 | 0 | 69/69 (100%) | 25 / 5 / 2 (trên 32) | 2124 / 108 | 7.84 / 10.34 |

### Tách lỗi tạm tính bằng khớp số (59 câu có con số trong đáp án)

| | Khớp đủ số | Không đúng |
|---|---|---|
| **Ngữ cảnh có đoạn đúng** | 45 — hệ thống đúng | 14 — **lỗi LLM** |
| **Ngữ cảnh không có đoạn đúng** | 0 — đúng dù thiếu căn cứ | 0 — **lỗi truy hồi** |

> Khớp số chỉ là chỉ số phụ: đúng số chưa chắc đúng ý (vd đúng số nhưng sai khóa), và câu định tính không chấm được. Kết luận dùng bảng của người chấm ở mục 6.

## 5. Lưu kết quả, manifest và xuất file chấm tay


In [6]:
import csv, importlib.metadata as im, platform

manifest = {
    "thoi_diem": time.strftime("%Y-%m-%d %H:%M:%S"), "lenh": "notebooks/danh_gia_tra_loi_dev.ipynb",
    "nhanh": BRANCH, "commit": COMMIT, "co_thay_doi_chua_commit": bool(chua_commit),
    "cau_hinh": cfg.mo_ta(), "llm": cfg.llm_model_name,
    "ollama": OLLAMA_BASE_URL, "model_server": {"url": MODEL_SERVER_URL, "health": SERVER_HEALTH},
    "may_local": {"python": platform.python_version(), "llama-index-core": im.version("llama-index-core")},
    "bo_cau_hoi": {bo: sum(1 for d in ket if d["bo"] == bo) for bo in BO_TRA_LOI},
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "ket-qua.json").write_text(json.dumps(ket, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT / "ket-qua-tu-dong.md").write_text(
    f"# Câu trả lời cấu hình chốt — {manifest['thoi_diem']}, commit `{COMMIT}`, {cfg.llm_model_name}\n\n" + MD_TU_DONG + "\n",
    encoding="utf-8")

COT = ["bo", "id", "question", "pham_vi", "dap_an", "tra_loi", "nguon", "ngu_canh_co_doan_dung", "khop_su_kien", "tu_choi",
       "ket_qua", "ly_do", "nguoi_cham"]
mau = OUT / "cham-tay.csv"
if mau.exists():
    print(f"Giữ nguyên {mau.name} (không ghi đè file có thể đã chấm)")
else:
    with mau.open("w", encoding="utf-8-sig", newline="") as fh:  # utf-8-sig: Excel/Google Sheets đọc đúng tiếng Việt
        w = csv.DictWriter(fh, fieldnames=COT)
        w.writeheader()
        for d in ket:
            w.writerow({**{c: d.get(c) for c in COT}, "nguon": "; ".join(d.get("nguon") or []),
                        "ket_qua": "loi" if d["ket_qua"] == "loi" else "", "ly_do": "", "nguoi_cham": ""})
print(f"Đã ghi {OUT.relative_to(ROOT)}/: manifest.json, ket-qua.json (có evidence), ket-qua-tu-dong.md, cham-tay.csv\n"
      "Chấm tay: mỗi người sao cham-tay.csv thành cham-tay-<tên>.csv trong cùng thư mục, điền cột ket_qua bằng một nhãn "
      "dung / mot_phan / sai / tu_choi_sai / bia (xem docs/rubric-cham-tra-loi.md; xem evidence trong ket-qua.json), "
      "rồi chạy ô cuối.")


Đã ghi eval/results/danh-gia-tra-loi-2026-10-23-2336-a6e63eb-chua-commit/: manifest.json, ket-qua.json (có evidence), ket-qua-tu-dong.md, cham-tay.csv
Chấm tay: mỗi người sao cham-tay.csv thành cham-tay-<tên>.csv trong cùng thư mục, điền cột ket_qua bằng một nhãn dung / mot_phan / sai / tu_choi_sai / bia (xem docs/rubric-cham-tra-loi.md; xem evidence trong ket-qua.json), rồi chạy ô cuối.


## 6. Tổng hợp sau khi chấm tay

Chạy sau khi đã có `cham-tay-<tên>.csv` (mỗi người một file) trong thư mục kết quả. Kernel mới thì đặt `TEN_LAN_CHAY` và chạy ô
tham số trước. Ra: tỉ lệ đúng theo từng người và từng bộ, bảng tách lỗi truy hồi/LLM, Cohen κ giữa hai người và giữa chỉ số
tự động với người chấm, ghi `tong-hop.md`.


In [7]:
import csv
from collections import Counter
from IPython.display import Markdown, display
from metrics import cohen_kappa
from tra_loi_dev import KET_QUA_HOP_LE

ket_luu = {(d["bo"], d["id"]): d for d in json.loads((OUT / "ket-qua.json").read_text(encoding="utf-8"))}
cham = {}  # người -> {(bộ, id): nhãn}
for f in sorted(OUT.glob("cham-tay-*.csv")):
    with f.open(encoding="utf-8-sig") as fh:
        rows = list(csv.DictReader(fh))
    nhan = {(r["bo"], r["id"]): r["ket_qua"].strip() for r in rows}
    thieu = [k for k, v in nhan.items() if not v]
    sai = sorted({v for v in nhan.values() if v and v not in KET_QUA_HOP_LE})
    if sai:
        raise SystemExit(f"{f.name}: nhãn không hợp lệ {sai}; chỉ nhận {KET_QUA_HOP_LE}")
    if thieu:
        print(f"{f.name}: còn {len(thieu)} câu chưa chấm — bỏ qua các câu này")
    cham[f.stem.removeprefix("cham-tay-")] = {k: v for k, v in nhan.items() if v}
if not cham:
    raise SystemExit("Chưa có file cham-tay-<tên>.csv nào trong " + str(OUT))

DUNG = {"dung"}  # "mot_phan" tính riêng, không gộp vào đúng
phan = []
for nguoi, nhan in cham.items():
    dong = [f"### Người chấm: {nguoi}", "| Bộ | Số câu đã chấm | dung | mot_phan | sai | bia | tu_choi_sai | loi |",
            "|---|---|---|---|---|---|---|---|"]
    for bo in BO_TRA_LOI:
        c = Counter(v for (b, _), v in nhan.items() if b == bo)
        dong.append(f"| {TEN_BO.get(bo, bo)} | {sum(c.values())} | " + " | ".join(
            str(c[x]) for x in ("dung", "mot_phan", "sai", "bia", "tu_choi_sai", "loi")) + " |")
    phan.append("\n".join(dong))
    ds = [{**ket_luu[k], "_nguoi": v} for k, v in nhan.items() if v != "loi" and k in ket_luu]
    phan.append(f"Tách lỗi theo {nguoi} ({len(ds)} câu):\n\n" + bang_tach_loi(ds, lambda d: d["_nguoi"] in DUNG, "Đúng (dung)"))

ten = list(cham)
if len(ten) >= 2:
    a, b = cham[ten[0]], cham[ten[1]]
    chung = sorted(set(a) & set(b))
    phan.append(f"### Đồng thuận giữa {ten[0]} và {ten[1]} ({len(chung)} câu chung)\n\n"
                f"- κ trên đủ nhãn: {cohen_kappa([a[k] for k in chung], [b[k] for k in chung]):.2f}\n"
                f"- κ đúng/không đúng: {cohen_kappa([str(a[k] in DUNG) for k in chung], [str(b[k] in DUNG) for k in chung]):.2f}\n"
                f"- Câu lệch nhãn (cần họp chốt): " + ", ".join(f"{bo.split('_')[-1].removesuffix('.json')}#{i}"
                                                               for bo, i in chung if a[(bo, i)] != b[(bo, i)]))
nguoi0 = cham[ten[0]]
so = [k for k, v in nguoi0.items() if v != "loi" and k in ket_luu and ket_luu[k]["khop_su_kien"] in ("du", "thieu", "khong")]
if so:
    tu_dong = [str(ket_luu[k]["khop_su_kien"] == "du") for k in so]
    nguoi = [str(nguoi0[k] in DUNG) for k in so]
    phan.append(f"### Chỉ số tự động (khớp đủ số) so với {ten[0]} ({len(so)} câu có số)\n\n"
                f"κ = {cohen_kappa(tu_dong, nguoi):.2f}; trùng {sum(x == y for x, y in zip(tu_dong, nguoi))}/{len(so)}. "
                "κ thấp thì không dùng khớp số thay người chấm.")
MD_TONG_HOP = "\n\n".join(phan)
(OUT / "tong-hop.md").write_text(f"# Tổng hợp chấm tay — {TEN}\n\n" + MD_TONG_HOP + "\n", encoding="utf-8")
display(Markdown(MD_TONG_HOP))


SystemExit: Chưa có file cham-tay-<tên>.csv nào trong /mnt/data/1_Projects/sgu-rag-llamaindex/eval/results/danh-gia-tra-loi-2026-10-23-2336-a6e63eb-chua-commit

/mnt/data/1_Projects/sgu-rag-llamaindex/venv/lib64/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
